In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/math482-2024-2025-1-hw-05/sample_submission.csv
/kaggle/input/math482-2024-2025-1-hw-05/class_map.csv
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_398.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_12.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_173.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_277.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_116.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_360.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_334.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_102.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_304.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_354.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_127.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_46.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_126.png
/kaggle/input/math482-2024-2025-1-hw-05/test/test/test_100.png
/kag

In [2]:
# pip install  imutils

# Importing libararies to data reading and preprocessing

import cv2
import urllib
import itertools
import seaborn as sns
import random, os, glob
from imutils import paths
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from urllib.request import urlopen

# avoiding warnings

import warnings
warnings.filterwarnings("ignore")

# libs for model evaluation

from sklearn.metrics import confusion_matrix, classification_report

# libs for constructing the model

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.layers import Flatten, Conv2D, MaxPooling2D, Dense, Dropout, SpatialDropout2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, array_to_img
from tensorflow.keras.optimizers import Adam

In [3]:
target_size = (128,128)

shape_labels = {"circle":0, "triangle" : 1, "rectangle" : 2, "pentagon": 3,
                "hexagon": 4, "star":5}


In [4]:
from PIL import Image
from sklearn.model_selection import train_test_split
# Set paths

extraction_path = '/kaggle/input/math482-2024-2025-1-hw-05/'   # Replace with your preferred extraction folder

# Define train path
train_path = os.path.join(extraction_path, 'train/train')
class_folders = sorted(os.listdir(train_path))

# Load images and labels
images = []
labels = []
for class_id in class_folders:
    class_folder_path = os.path.join(train_path, class_id)
    for image_name in os.listdir(class_folder_path):
        image_path = os.path.join(class_folder_path, image_name)
        with Image.open(image_path) as img:
            img_array = np.array(img) / 255.0  # Normalize to [0, 1]
            images.append(img_array)
            labels.append(int(class_id))

# Convert lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

images, labels = shuffle(images, labels, random_state = 42)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, stratify=labels, random_state=42)

# One-hot encode labels
y_train = to_categorical(y_train, num_classes=6)
y_val = to_categorical(y_val, num_classes=6)


In [5]:
print(f"image size: {images.shape}")
print(f"labes class number: {len(np.unique(labels))} observation num: {len(labels)}")

image size: (1600, 128, 128, 3)
labes class number: 6 observation num: 1600


In [6]:
input_shape = (np.array(images[0]).shape[1], (np.array(images[0]).shape[1], 3))

In [7]:
# Define CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')  # 6 output classes
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

# Fit model with early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=30,
    callbacks=[early_stopping]
)


Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 27s 616ms/step - accuracy: 0.1895 - loss: 2.3998 - val_accuracy: 0.2094 - val_loss: 1.7862
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 25s 602ms/step - accuracy: 0.2094 - loss: 1.7899 - val_accuracy: 0.2031 - val_loss: 1.7865
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 628ms/step - accuracy: 0.1993 - loss: 1.7878 - val_accuracy: 0.2031 - val_loss: 1.7830
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 626ms/step - accuracy: 0.1811 - loss: 1.7890 - val_accuracy: 0.1937 - val_loss: 1.7840
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 622ms/step - accuracy: 0.1763 - loss: 1.7899 - val_accuracy: 0.2313 - val_loss: 1.7829
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 631ms/step - accuracy: 0.2170 - loss: 1.7903 - val_accuracy: 0.2125 - val_loss: 1.7826
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 617ms/step - accuracy: 0.1960 - loss: 1.7862 - val_accuracy: 0.2000 - val_loss: 1.7827
Epoch 8/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 26s 616ms/step - accuracy: 0.2001 - loss: 1.7853 - val_accu

In [8]:
# Evaluate on validation set
val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {val_accuracy:.2f}")

# Load test images for predictions
test_path = os.path.join(extraction_path, 'test/test')
test_images = []
test_filenames = []

for image_name in os.listdir(test_path):
    image_path = os.path.join(test_path, image_name)
    with Image.open(image_path) as img:
        img_array = np.array(img) / 255.0  # Normalize
        test_images.append(img_array)
        test_filenames.append(image_name)

test_images = np.array(test_images)

# Predict on test set
test_predictions = model.predict(test_images)
test_labels = np.argmax(test_predictions, axis=1)




10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 146ms/step - accuracy: 0.2064 - loss: 1.7724
Validation Accuracy: 0.20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 135ms/step


In [9]:
submission = pd.DataFrame({'filename': test_filenames, 'label': test_labels})


# Ensure filenames are sorted numerically
submission['numeric_filename'] = submission['filename'].str.extract(r'(\d+)').astype(int)  # Extract numeric part
submission = submission.sort_values('numeric_filename').drop(columns='numeric_filename')

# Reset index after sorting
submission = submission.reset_index(drop=True)

# Display the sorted DataFrame
submission


,filename,label
0,test_0.png,0
1,test_1.png,0
2,test_2.png,0
3,test_3.png,1
4,test_4.png,1
...,...,...
395,test_395.png,1
396,test_396.png,0
397,test_397.png,0
398,test_398.png,0


In [10]:
submission.to_csv('submission.csv', index=False)